# Notebook 02 — RAG Chain
**Goal:** Query the Pinecone corpus and get grounded answers from GPT-4o-mini.

By the end of this notebook you will have:
- Connected to your Pinecone index with ingested video transcripts
- Run a similarity search and inspected the retrieved chunks
- Built a RAG chain that answers questions grounded in transcript content
- Tested multiple query types (hooks, themes, specific claims)

**Prerequisite:** Run notebook 01 first to ingest at least one video.

## Step 1 — Environment check

In [ ]:
import sys
sys.path.append('..')

from src.utils.config import (
    OPENAI_API_KEY,
    OPENAI_LLM_MODEL,
    OPENAI_EMBEDDING_MODEL,
    PINECONE_API_KEY,
    PINECONE_INDEX_NAME,
    TOP_K_RESULTS,
)

print('✅ OpenAI key loaded:', OPENAI_API_KEY[:8] + '...')
print(f'✅ LLM model: {OPENAI_LLM_MODEL}')
print(f'✅ Embedding model: {OPENAI_EMBEDDING_MODEL}')
print(f'✅ Pinecone index: {PINECONE_INDEX_NAME}')
print(f'✅ Top-K results: {TOP_K_RESULTS}')

## Step 2 — Check Pinecone has vectors
Make sure notebook 01 ran successfully and vectors are in the index.

In [ ]:
from pinecone import Pinecone

pc = Pinecone(api_key=PINECONE_API_KEY)
index = pc.Index(PINECONE_INDEX_NAME)
stats = index.describe_index_stats()

vector_count = stats['total_vector_count']
print(f'Vectors in index: {vector_count}')
assert vector_count > 0, 'No vectors found — run notebook 01 first to ingest a video.'
print('✅ Index has data, ready to query.')

## Step 3 — Raw similarity search
Embed a question and retrieve the top-K chunks directly from Pinecone.

In [ ]:
from src.ingestion.embedder import embed_texts

QUESTION = 'What skincare routine does the creator recommend?'

query_embedding = embed_texts([QUESTION])[0]
print(f'Query embedding dimension: {len(query_embedding)}')

results = index.query(
    vector=query_embedding,
    top_k=TOP_K_RESULTS,
    include_metadata=True,
)

print(f'\nRetrieved {len(results["matches"])} chunks:\n')
for i, match in enumerate(results['matches'], 1):
    score = match['score']
    text = match['metadata'].get('text', '')[:200]
    title = match['metadata'].get('title', 'Unknown')
    print(f'[{i}] Score: {score:.4f} | Source: {title}')
    print(f'    {text}...')
    print()

## Step 4 — Build the RAG chain with LangChain
Use the PineconeVectorStore + ChatOpenAI to build a retrieval-augmented generation chain.

In [ ]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_pinecone import PineconeVectorStore
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# Embeddings + vectorstore
embeddings = OpenAIEmbeddings(
    model=OPENAI_EMBEDDING_MODEL,
    openai_api_key=OPENAI_API_KEY,
)
vectorstore = PineconeVectorStore(
    index_name=PINECONE_INDEX_NAME,
    embedding=embeddings,
)
retriever = vectorstore.as_retriever(search_kwargs={'k': TOP_K_RESULTS})

# LLM
llm = ChatOpenAI(
    model=OPENAI_LLM_MODEL,
    openai_api_key=OPENAI_API_KEY,
    temperature=0,
)

# RAG prompt
RAG_PROMPT = ChatPromptTemplate.from_template(
    """You are a Creative Intelligence Copilot. Answer the question based ONLY on the
following transcript excerpts. If the answer is not in the context, say so.
Always cite which video/source the information comes from.

Context:
{context}

Question: {question}

Answer:"""
)


def format_docs(docs):
    return '\n\n---\n\n'.join(
        f"[Source: {d.metadata.get('title', 'Unknown')}]\n{d.page_content}"
        for d in docs
    )


rag_chain = (
    {'context': retriever | format_docs, 'question': RunnablePassthrough()}
    | RAG_PROMPT
    | llm
    | StrOutputParser()
)

print('✅ RAG chain built successfully.')

## Step 5 — Test the RAG chain
Run a few different question types to see how the chain performs.

In [ ]:
question = 'What skincare routine or products does the creator recommend?'
print(f'Q: {question}\n')
answer = rag_chain.invoke(question)
print(f'A: {answer}')

In [ ]:
question = 'What hook or opening line does the creator use to grab attention?'
print(f'Q: {question}\n')
answer = rag_chain.invoke(question)
print(f'A: {answer}')

In [ ]:
question = 'Are there any medical or treatment claims in the transcript?'
print(f'Q: {question}\n')
answer = rag_chain.invoke(question)
print(f'A: {answer}')

## Step 6 — Use the query_corpus tool directly
This is the same function the agent uses internally.

In [ ]:
from src.agent.tools import query_corpus_tool

result = query_corpus_tool.invoke('What are the main themes discussed in the videos?')
print(result)

## Notes

**Retrieval quality depends on the corpus.** If answers seem generic, ingest more videos with notebook 01.

**Top-K tuning:** The default is 5 chunks. Increase `TOP_K_RESULTS` in `.env` for broader context, but watch for token limits.

**Next step:** Move to notebook 03 to test the compliance checker on real transcript content.